# Rotation Correlation Curve Analysis

Standalone debug viewer for the **2D SOFT rotation correlation** (all angles in **radians**). It reads the CSV dumps that the fs2d debug run already produces (no C++ changes needed).

## Workflow
1. Run a registration with the debug flag on (service or test executable, `useDirect=true` for the coefficient files).
2. In this notebook click **Run All**.
3. Zoom into any region of the curve (rangeslider at the bottom, or the lo/hi sliders) to inspect its curvature.
4. The last section fits the curve with the **true kernel** and runs a **hidden-component scan** that can reveal *shoulder/plateau* features which classical peak detection cannot see (e.g. the ~0.55 rad / GT region of the earlier pair 285->290).

## Files read from `./data`
| file | content |
|---|---|
| `rotationCorrelation1D.csv` | correlation curve `[index, angle(rad), normalizedCorrelation]` |
| `rotationPeaks.csv` | detected peaks `[angle, peakCorrelation, covariance, levelPotential, index]` |
| `registration_meta.csv` | pair + estimated rotation + GT error (one row) |
| `dataForReadIn.csv` | parameters (N, thresholds, numAngles, numTotalSolutions) |
| `sigCoefR/I_1angle.csv`, `patCoefR/I_1angle.csv` | spherical-harmonic coefficients (written by the `useDirect=true` debug path) -> true kernel |

Requirements: `numpy`, `plotly`, `ipywidgets` (same env as `plot_2d_registration.ipynb`).


In [ ]:
"""Imports + data location."""
import os
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown

# Data directory: this notebook lives in plotting_results/2d, debug output in ./data
DATA_DIR = os.path.join(os.getcwd(), "data")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data"
print("DATA_DIR:", DATA_DIR)


In [ ]:
"""Robust loaders (tolerate headers, tab/comma/space delimiters, trailing non-numeric tokens)."""

def load_lines(fname):
    """All parsed rows (list of lists of floats); headers / non-numeric rows skipped."""
    path = os.path.join(DATA_DIR, fname)
    if not os.path.isfile(path):
        return None
    out = []
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("#"):
                continue
            vals = []
            for t in ln.replace(",", " ").split():
                try:
                    vals.append(float(t))
                except ValueError:
                    break
            if vals:
                out.append(vals)
    return out

def load_csv(fname):
    """2D array using the dominant column count (e.g. registration_meta keeps its 13 numeric cols)."""
    rows = load_lines(fname)
    if not rows:
        return None
    counts = [len(r) for r in rows]
    modal = max(set(counts), key=counts.count)
    rows = [r for r in rows if len(r) == modal]
    return np.array(rows) if rows else None

def status(name, arr):
    print("  [%s] %s%s" % ("ok" if arr is not None else "MISSING", name,
                           "  (%d rows)" % arr.shape[0] if arr is not None and arr.ndim == 2 else ""))


In [ ]:
"""Load everything and print a summary (angles in rad)."""
print("Loading debug files from", DATA_DIR)
curve = load_csv("rotationCorrelation1D.csv")   # [index, angle(rad), normalizedCorrelation]
peaks = load_csv("rotationPeaks.csv")           # [angle, peakCorrelation, covariance, levelPotential, index]
meta  = load_csv("registration_meta.csv")       # frame1 frame2 rot_angle_deg tx ty ...
cfg   = load_lines("dataForReadIn.csv")         # parameter rows of mixed width

status("rotationCorrelation1D.csv", curve)
status("rotationPeaks.csv", peaks)
status("registration_meta.csv", meta)
import time as _t
for _f in ("rotationCorrelation1D.csv", "rotationPeaks.csv", "registration_meta.csv"):
    _p = os.path.join(DATA_DIR, _f)
    if os.path.isfile(_p):
        print("    mtime %s: %s" % (_f, _t.strftime("%Y-%m-%d %H:%M:%S", _t.localtime(os.path.getmtime(_p)))))

# --- parameters (dataForReadIn: 6-col metadata row + per-angle 3-col rows) ---
N = num_angles = num_total = None
level_thresh = None
if cfg:
    for r in cfg:
        if len(r) == 6:
            N = int(r[0]); _cn = int(r[1]); _cell = r[2]
            level_thresh = r[3]; num_angles = int(r[4]); num_total = int(r[5])
            break
print("\nParameters: N=%s  level_potential=%s  numAngles=%s  numTotalSolutions=%s" % (N, level_thresh, num_angles, num_total))

# --- ground truth (derived: GT = estimated rotation + GT error), converted to rad ---
gt_rad, pair = None, None
if meta is not None and len(meta) >= 1:
    pair = (int(meta[0, 0]), int(meta[0, 1]))
    est_deg = float(meta[0, 2])
    gt_err_deg = float(meta[0, 7]) if meta.shape[1] > 7 else 0.0
    gt_rad = np.deg2rad(est_deg + gt_err_deg)
    print("Pair: %d -> %d   estimated rotation: %.4f rad   GT error: %.4f rad   -> GT: %.4f rad"
          % (pair[0], pair[1], np.deg2rad(est_deg), np.deg2rad(gt_err_deg), gt_rad))
else:
    print("No registration_meta.csv -> ground truth marker will not be shown.")

# --- detected peaks ---
if peaks is not None:
    print("\nDetected rotation peaks (from rotationPeaks.csv):")
    print("  %10s %11s %13s" % ("angle(rad)", "correlation", "levelPotential"))
    for p in peaks:
        print("  %10.4f %11.4f %13.4f" % (p[0], p[1], p[3]))
    print("  Note: every peak has an antipodal copy at +pi rad (the curve is pi-periodic).")
    print("  Physical rotations are angles modulo pi rad.")


In [ ]:
"""Build the (interactive) correlation curve figure."""
import plotly.graph_objects as go

fig1 = go.Figure()   # plain Figure: no anywidget dependency
if curve is None:
    print("No correlation curve available.")
else:
    x, c = curve[:, 1], curve[:, 2]

    # robust fold: uniform [0, 2pi) grid with an even number of samples
    n = len(x)
    step = np.median(np.diff(x))
    uniform = np.allclose(np.diff(x), step, atol=1e-9)
    if not uniform or not np.isclose(x[0], 0.0, atol=1e-6):
        xg = np.linspace(0.0, 2 * np.pi, n)
        c = np.interp(xg, x, c)
        x = xg
    n = len(x) - (len(x) % 2)
    x, c = x[:n], c[:n]
    half = n // 2
    fold_x = x[:half]
    fold_c = (c[:half] + c[half:]) / 2.0

    fig1.add_trace(go.Scatter(x=x, y=c, name="correlation C(theta)",
                              line=dict(color="steelblue", width=2),
                              hovertemplate="%{x:.4f} rad<br>corr %{y:.4f}<extra></extra>"))
    fig1.add_trace(go.Scatter(x=fold_x, y=fold_c, name="folded [0,pi) = (C+C(pi))/2",
                              line=dict(color="orange", width=1.5, dash="dot"), visible="legendonly"))

    if peaks is not None:
        fig1.add_trace(go.Scatter(x=peaks[:, 0], y=peaks[:, 1], mode="markers+text",
                                  name="detected peaks",
                                  marker=dict(symbol="x", size=12, color="red", line=dict(width=2)),
                                  text=["%.4f rad" % a for a in peaks[:, 0]],
                                  textposition="top center", textfont=dict(size=10, color="red")))

    if gt_rad is not None:
        fig1.add_vline(x=gt_rad, line=dict(color="green", dash="dash", width=1.5),
                       annotation_text="GT %.4f rad" % gt_rad, annotation_position="top left")

    cm = c.min() if len(c) else 0.0
    fig1.update_layout(
        title="Rotation correlation curve, pair %d -> %d" % (pair[0], pair[1]) if pair else "Rotation correlation curve",
        height=520,
        legend=dict(orientation="h", y=1.12, font=dict(size=11)),
        xaxis=dict(title="rotation angle (rad)", rangeslider=dict(visible=True),
                   range=[0, 2 * np.pi], showgrid=True,
                   tickvals=[0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi],
                   ticktext=["0", "pi/2", "pi", "3pi/2", "2pi"]),
        yaxis=dict(title="normalized correlation", range=[min(cm, 0) - 0.05, 1.05]),
        margin=dict(t=80))
    print("Tip: drag the rangeslider at the bottom to zoom, or use the sliders below.")


In [ ]:
"""Zoom helpers: set the visible x-range of the figure above (re-rendered via an Output widget)."""
lo = widgets.FloatSlider(value=0.0, min=0.0, max=2 * np.pi, step=0.01, description="lo (rad)")
hi = widgets.FloatSlider(value=2 * np.pi, min=0.0, max=2 * np.pi, step=0.01, description="hi (rad)")
btn = widgets.Button(description="Apply zoom")
reset = widgets.Button(description="Reset")
out = widgets.Output()

def _redraw():
    with out:
        out.clear_output(wait=True)
        display(fig1)

def _apply(_b):
    a, b = sorted([lo.value, hi.value])
    fig1.update_xaxes(range=[a, b])
    _redraw()

def _reset(_b):
    fig1.update_xaxes(range=[0, 2 * np.pi])
    _redraw()

btn.on_click(_apply)
reset.on_click(_reset)
_redraw()
display(widgets.VBox([widgets.HBox([lo, hi]), widgets.HBox([btn, reset]), out]))


## Kernel fit & hidden-component scan

**What is the "true kernel"?** A matched rotation peak does not look Gaussian in this pipeline: its shape is the autocorrelation of the resampled reference descriptor (a band-limited bump with tails), computable exactly and for free from the SH coefficients of the debug dump:

$$K(\theta) = \sum_{m \neq 0} \left(\sum_{\ell} |\hat b_{\ell m}|^2\right) \cos(m\theta)$$

**Why fold?** The curve is (nearly) exactly pi-periodic: $C(\theta+\pi)=C(\theta)$. Everything lives in $[0, \pi)$; antipodal copies need no special handling.

**The scan:** known components (the persistence peaks, folded mod $\pi$) are fitted with non-negative amplitudes inside a window (default: around GT, +/- `WIN_HALF_RAD`). Then a grid search looks for ONE additional component in that window - a *shoulder/plateau* that is not a local maximum and is invisible to classical peak detection (this is how the ~0.55 rad / GT region of the earlier pair 285->290 appears, and why that pair's plateau at GT was missed). The residual improvement quantifies the evidence.

Tune `WIN_HALF_RAD`, `WIN_CENTER_RAD`, `MIN_SEP_RAD`, `KNOWN_MARGIN_RAD` in the next cell if needed.


In [ ]:
"""True kernel from the SH coefficients of the reference (scan 2) descriptor + scan parameters (in rad)."""
# --- fit parameters (all in radians) --------------------------------
COARSE_RAD = 0.01        # hidden-component scan grid step (~0.57 deg)
FINE_RAD = 0.0004        # local refinement step (~0.02 deg)
MIN_SEP_RAD = 0.1        # min separation between components (~5.7 deg)
WIN_CENTER_RAD = None    # scan window center (default: GT mod pi)
WIN_HALF_RAD = 0.35      # scan window half width (~20 deg)
KNOWN_MARGIN_RAD = 0.26  # persistence peaks within +/-this of the window are known components

c2R = load_csv("patCoefR_1angle.csv")
c2I = load_csv("patCoefI_1angle.csv")
c1R = load_csv("sigCoefR_1angle.csv")
c1I = load_csv("sigCoefI_1angle.csv")

def build_kernel_acf(cR, cI, theta):
    """Autocorrelation kernel of a descriptor from its SH coefficients:
    K(theta) = sum_{m!=0} (sum_l |c_lm|^2) cos(m theta).
    The coefficient array uses the alm layout of softRegistrationClass.cpp
    (see the almIdx formula there); the file is a raw dump of that array."""
    bw = int(round(np.sqrt(len(cR))))
    bigL = bw - 1
    Q = np.zeros(2 * bw)
    for l in range(bw):
        for m in range(-l, l + 1):
            if m >= 0:
                idx = m * (bigL + 1) - m * (m - 1) // 2 + (l - m)
            else:
                idx = bigL * (bigL + 3) // 2 + 1 + (bigL + m) * (bigL + m + 1) // 2 + (l - abs(m))
            Q[m + bw] += cR[idx] ** 2 + cI[idx] ** 2
    mpos = np.arange(1, bw)                       # Q is symmetric in m (descriptor is real)
    K = 2.0 * (Q[mpos + bw] @ np.cos(np.outer(mpos, theta)))
    return K

def build_kernel_empirical(theta):
    """Fallback: measure the peak shape from the strongest peak of the observed
    folded curve itself (mirrored, 0.9 rad wide)."""
    Fk = fold_c
    i0 = int(np.argmax(Fk))
    W = int(round(0.9 / (theta[1] - theta[0])))
    seg = Fk[max(0, i0 - W):i0 + W + 1]
    K = np.zeros(len(theta))
    for j in range(W + 1):
        v = (seg[W - j] + (seg[W + j] if W + j < len(seg) else 0.0)) / 2.0
        K[j] = v
        if j > 0:
            K[len(theta) - j - 1] = v
    return K

if curve is None:
    print("Kernel panel skipped: no correlation curve.")
    K = None
elif c2R is not None and c2I is not None:
    print("Building true kernel K from scan-2 descriptor (patCoef, %d coefficients)" % len(c2R))
    K = build_kernel_acf(c2R[:, 0], c2I[:, 0], x)
    K = (K - K.min()) / (K.max() - K.min())
elif c1R is not None and c1I is not None:
    print("WARNING: patCoef missing -> using scan-1 descriptor (sigCoef) as kernel.")
    K = build_kernel_acf(c1R[:, 0], c1I[:, 0], x)
    K = (K - K.min()) / (K.max() - K.min())
else:
    print("WARNING: no coefficient files -> measuring the kernel empirically from the dominant peak.")
    K = build_kernel_empirical(x)
    K = (K - K.min()) / (K.max() - K.min())


In [ ]:
"""Windowed hidden-component scan on the folded curve (angles in rad).

Known components (persistence peaks folded mod pi, inside the window + margin)
are kept fixed; a search grid inside the window looks for ONE additional
component (a shoulder/plateau), reporting position, amplitude, residual
improvement and whether classical peak detection could see it."""
mus_known, amps_known = [], []
hidden_result = None
scan_rad, scan_resid = [], []

def circ_dist(a, b, period=np.pi):
    d = np.abs(np.asarray(a) - b) % period
    return np.minimum(d, period - d)

def kval(a):
    w = np.mod(np.asarray(a), 2 * np.pi)
    return np.interp(np.minimum(w, 2 * np.pi - w), x, K)   # K is even on [0, 2pi)

def design_matrix(angles, th):
    return np.column_stack([kval(th - a) for a in angles] + [np.ones(len(th))])

def fit_nonneg(th, Fw, angles_in):
    """Least squares with non-negative component amplitudes, affine baseline.
    Returns (coef, r, kept_angles); negative-amplitude components are dropped."""
    kept = list(angles_in)
    while True:
        A = design_matrix(kept, th)
        coef, res, *_ = np.linalg.lstsq(A, Fw, rcond=None)
        r = res[0] if len(res) else float(np.sum((A @ coef - Fw) ** 2))
        amps = coef[:-1]
        if len(amps) == 0 or np.all(amps >= 0):
            return coef, r, kept
        del kept[int(np.argmin(amps))]

if K is not None:
    th, F = fold_x, fold_c                         # folded grid [0, pi)
    known_all = [float(p) % np.pi for p in peaks[:, 0]] if peaks is not None else []

    ctr = WIN_CENTER_RAD
    if ctr is None:
        ctr = float(gt_rad % np.pi) if gt_rad is not None else 0.5
    lo, hi = ctr - WIN_HALF_RAD, ctr + WIN_HALF_RAD
    win = (th > lo) & (th < hi)
    thw, Fw = th[win], F[win]

    win_samp = np.arange(lo, hi, 0.01) % np.pi    # window on the folded circle
    mus_known = [m for m in known_all if np.min(circ_dist(m, win_samp)) < KNOWN_MARGIN_RAD]
    mus_known = list(dict.fromkeys(mus_known))
    coef_k, r_k, kept_k = fit_nonneg(thw, Fw, mus_known)
    mus_known = kept_k
    amps_known = coef_k[:-1]

    print("Hidden-component scan window: [%.4f, %.4f] rad" % (lo, hi))
    print("Known components (persistence peaks, folded): %s"
          % [round(m, 4) for m in mus_known])
    print("Baseline-only residual in window: %.6f" % r_k)

    cand = np.arange(lo + COARSE_RAD, hi - COARSE_RAD, COARSE_RAD)
    if len(cand) == 0:
        print("Window too small; increase WIN_HALF_RAD.")
        hidden_result = None
    else:
        resids = []
        for mu in cand:
            if any(circ_dist(mu, m) < MIN_SEP_RAD for m in mus_known):
                resids.append(np.inf)
                continue
            coef, r, kept = fit_nonneg(thw, Fw, mus_known + [mu])
            resids.append(r if kept == mus_known + [mu] else np.inf)
        scan_rad, scan_resid = list(cand), resids
        resids = np.array(resids)
        ibest = int(np.argmin(resids))
        if np.isinf(resids[ibest]):
            print("No valid hidden component found in the window.")
        else:
            mu0 = cand[ibest]
            r0 = resids[ibest]
            for mu_c in np.arange(mu0 - 0.044, mu0 + 0.044, FINE_RAD):
                if any(circ_dist(mu_c, m) < MIN_SEP_RAD for m in mus_known):
                    continue
                coef, r, kept = fit_nonneg(thw, Fw, mus_known + [mu_c])
                if kept != mus_known + [mu_c]:
                    continue
                if r < r0:
                    r0, mu0 = r, mu_c
            coef, r, kept = fit_nonneg(thw, Fw, mus_known + [mu0])
            hA = float(coef[len(mus_known)])      # hidden amplitude
            lm = [th[i] for i in range(1, len(th) - 1) if F[i] > F[i - 1] and F[i] > F[i + 1]]
            shoulder = not any(circ_dist(mu0, m) < 0.026 for m in lm)
            hidden_result = dict(mu=float(mu0), amp=hA, resid0=r_k, resid1=r,
                                 shoulder=bool(shoulder), coef=coef, mus_all=kept,
                                 lo=lo, hi=hi)
            print("\nHidden component found:  mu = %.4f rad (mod pi: %.4f)   amplitude A = %.3f"
                  % (mu0, mu0 % np.pi, hA))
            print("  residual: %.6f -> %.6f   (%.1f x improvement)" % (r_k, r, r_k / r))
            print("  -> %s" % ("SHOULDER/PLATEAU (invisible to maxima-based peak detection)"
                               if shoulder else "local maximum (classical detection would find it)"))
            print("\nTop-3 scan candidates (before refinement):")
            for i in np.argsort(resids)[:3]:
                print("   %8.4f rad   residual %.6f" % (cand[i], resids[i]))
else:
    print("Kernel fit skipped (K not available, see previous cell).")


In [ ]:
"""Figure 2: folded curve, scan window, kernel model, components, residual (rad)."""
import plotly.graph_objects as go

if K is not None and hidden_result is not None:
    thd = th                                        # folded grid in rad
    mus_all = hidden_result["mus_all"]
    coef_all = hidden_result["coef"]
    model_all = design_matrix(mus_all, th) @ coef_all

    fig2 = go.Figure()
    fig2.add_vrect(x0=hidden_result["lo"], x1=hidden_result["hi"],
                   fillcolor="lightgray", opacity=0.25, line_width=0,
                   annotation_text="scan window", annotation_position="top left")
    fig2.add_trace(go.Scatter(x=thd, y=F, name="folded observed", line=dict(color="black", width=2)))
    fig2.add_trace(go.Scatter(x=thd, y=model_all, name="kernel model", line=dict(color="steelblue", width=1.8)))
    for j, mu in enumerate(mus_all):
        comp = coef_all[j] * kval(th - mu)
        fig2.add_trace(go.Scatter(x=thd, y=comp, line=dict(dash="dash", width=1.2),
                                  name="comp %d @ %.4f rad (A=%.2f)" % (j + 1, mu % np.pi, coef_all[j])))
        fig2.add_vline(x=mu % np.pi, line=dict(color="steelblue", dash="dot", width=1))
    if hidden_result["mu"] not in mus_all:
        mu = hidden_result["mu"]
        comp = hidden_result["amp"] * kval(th - mu)
        fig2.add_trace(go.Scatter(x=thd, y=comp, line=dict(dash="dash", width=1.4, color="darkred"),
                                  name="HIDDEN comp @ %.4f rad (A=%.2f)" % (mu % np.pi, hidden_result["amp"])))
        fig2.add_vline(x=mu % np.pi, line=dict(color="darkred", dash="dot", width=1))
    res = F - model_all
    fig2.add_trace(go.Scatter(x=thd, y=res, name="residual", yaxis="y2",
                              line=dict(color="red", width=1.2, dash="dot"),
                              hovertemplate="%{x:.4f} rad<br>resid %{y:.5f}<extra></extra>"))
    if peaks is not None:
        prad = np.mod(peaks[:, 0], np.pi)
        fig2.add_trace(go.Scatter(x=prad, y=np.interp(prad, thd, F), mode="markers",
                                  name="persistence peaks (mod pi)",
                                  marker=dict(symbol="x", size=10, color="red", line=dict(width=2)),
                                  hovertemplate="%{x:.4f} rad<extra></extra>"))
    if gt_rad is not None:
        fig2.add_vline(x=gt_rad % np.pi, line=dict(color="green", dash="dash"),
                       annotation_text="GT %.4f rad" % gt_rad, annotation_position="top right")
    fig2.update_layout(title="Folded correlation + kernel fit (windowed)",
                       height=560,
                       legend=dict(orientation="h", y=1.12, font=dict(size=10)),
                       xaxis=dict(title="rotation angle (rad)", range=[0, np.pi],
                                  tickvals=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi],
                                  ticktext=["0", "pi/4", "pi/2", "3pi/4", "pi"]),
                       yaxis=dict(title="normalized correlation"),
                       yaxis2=dict(title="residual", overlaying="y", side="right", showgrid=False),
                       margin=dict(t=80))
    fig2.show()
else:
    print("Figure 2 skipped (no fit results).")


In [ ]:
"""Combined summary: persistence peaks vs. hidden-component scan vs. GT (rad)."""
if K is not None:
    print("=== Persistence peaks (rotationPeaks.csv, folded mod pi) ===")
    for p in peaks:
        print("  %9.4f rad  corr=%.3f  levelPot=%.4f" % (p[0] % np.pi, p[1], p[3]))
    if hidden_result is not None:
        h = hidden_result
        print("\n=== Hidden-component scan (window [%.4f, %.4f] rad) ===" % (h["lo"], h["hi"]))
        print("  best hidden component: %.4f rad (mod pi: %.4f),  A = %.3f,  residual %.4f -> %.4f (%.1fx)"
              % (h["mu"], h["mu"] % np.pi, h["amp"], h["resid0"], h["resid1"], h["resid0"] / h["resid1"]))
        print("  type: %s" % ("SHOULDER/PLATEAU" if h["shoulder"] else "local maximum"))
        if gt_rad is not None:
            near = circ_dist(gt_rad % np.pi, h["mu"]) < 0.035
            print("  GT rotation %.4f rad -> hidden component %s" % (gt_rad, "within 0.035 rad" if near else "NOT near GT"))
    print("\nInterpretation: a hidden component classified as SHOULDER/PLATEAU sits on the")
    print("flank of a dominant peak - classical peak detection cannot see it, the kernel")
    print("fit can. Check the residual trace in Figure 2 for remaining structure.")
else:
    print("Summary skipped (no fit results).")


## Notes & pitfalls

- **pi-periodicity:** the resampled Fourier magnitude is even in azimuth, so $C(\theta)$ is (bit-)exactly periodic in $\pi$. The folded trace is lossless; angles are physical rotations modulo $\pi$ rad.
- **GT derivation:** `registration_meta.csv` stores the *estimated* rotation and the GT *error* (in degrees); GT = estimated + error, displayed in radians. Verify the pair is the one you ran.
- **Gaussian fits will mislead:** the true peak kernel is NOT Gaussian (band-limited correlation: sharp core + tails + side lobes, e.g. a strong lobe near $\pi/2$ rad in one earlier pair). A Gaussian mixture invents spurious components there. Use the kernel from the coefficients.
- **Shoulders are invisible to peak detectors** (local-max / persistence alike): they are not local maxima. Only model fitting (kernel subtraction) reveals them.
- **Resolution limit:** two rotations closer than ~ the kernel core width (roughly 0.05-0.15 rad) cannot be separated; the scan enforces `MIN_SEP_RAD` separation from known components.
- **The kernel is per scan pair:** it is the autocorrelation of the *reference* (scan 2) descriptor. Recompute it for each debug run (the notebook does this automatically).
- Missing files? Run fs2d with debug + `useDirect=true` to get the coefficient files; without them the notebook falls back to an empirical kernel (dominant peak shape).
